<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Charts for rates and counts

**[Charts for rates and counts](https://learning.nextia-ai.com/courses/anomaly-detection/m03/charts-for-rates-and-counts/)** · Anomaly Detection: Find What Is Unusual · Module 3, lesson 3 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** draw a p-chart for a rate whose daily volume changes, a c-chart and a u-chart for counts, and a Laney chart when the data vary more than the simple model expects (overdispersion).

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (less than 1 MB). No account. |
| **You should know** | The I-MR chart and phase I, from [Control charts: is this variation normal?](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/). |
| **You do not need** | A statistics package: every limit is one line of pandas. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m03/charts-for-rates-and-counts/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M03-L03-charts-for-rates-and-counts/charts-for-rates-and-counts-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`refunds_daily.csv`, `orders.csv`, `tickets_daily.csv`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "refunds_daily.csv": "04c6384d1fbe8da908a739edc084eabc413581d75fce7cd84dcf7a90988996a6",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The same rate on a small day and a big day

A refund rate of 3% on a day with 300 orders is 9 refunds. Three more refunds by chance move it to 4%. On a day with 1,200 orders, 3% is 36 refunds, and chance moves it much less. So the limits of a rate must depend on the size of the day.

The **p-chart** does this. For a rate p̄ (p-bar) and a day with n orders, one sigma is √(p̄(1 − p̄)/n), and the limits are 3 of these sigmas from p̄.

> **Predict.** Four days, p̄ = 2.14%. A: 300 orders, 9 refunds (3.00%). B: 1,200 orders, 36 refunds (3.00%). C: 1,200 orders, 45 refunds (3.75%). D: 300 orders, 13 refunds (4.33%). Which days are above their upper limit? Run the cell.

> **Check.** You should see `True` only for day C.

In [ ]:
pbar = 0.0214                                          # the refund rate of phase I
days = pd.DataFrame({"orders": [300, 1200, 1200, 300], "refunds": [9, 36, 45, 13]}, index=list("ABCD"))
days["rate"] = days.refunds / days.orders
days["sigma"] = np.sqrt(pbar * (1 - pbar) / days.orders)     # smaller when the day has more orders
days["ucl"] = pbar + 3 * days.sigma
days["outside"] = days.rate > days.ucl
print((days.assign(rate=days.rate * 100, ucl=days.ucl * 100)).round(3).to_string())

Day D has the highest rate, and it is inside its limit: on a small day, 4.65% is still normal noise. Day C has a lower rate, but on a big day its limit is 3.39%.

## 2. The p-chart of Larkfield's refunds

Run the next cell. It computes p̄ from phase I (April to December 2024), one sigma per day, and the days outside the limits.

> **Check.** You should see p-bar 0.02141 and the UCL 0.0465 at 300 orders.

In [ ]:
daily = pd.read_csv("data/refunds_daily.csv", parse_dates=["date"]).set_index("date")
x = daily.dropna()                                      # 2025-10-19 has no orders: the feed missed it
p1 = x.loc["2024-04-01":"2024-12-31"]
pbar = p1.refunds.sum() / p1.orders.sum()                # all refunds of phase I / all its orders
sigma_day = np.sqrt(pbar * (1 - pbar) / x.orders)        # one sigma per day
ucl, lcl = pbar + 3 * sigma_day, (pbar - 3 * sigma_day).clip(lower=0)
print(f"p-bar {pbar:.5f}; UCL at 300 orders {pbar + 3 * np.sqrt(pbar * (1 - pbar) / 300):.4f}, "
      f"at 1,000 orders {pbar + 3 * np.sqrt(pbar * (1 - pbar) / 1000):.4f}")
outside = (x.refund_rate > ucl) | (x.refund_rate < lcl)
print("p-chart, days outside:", [str(d.date()) for d in x.index[outside]])

Run the next cell. It draws October to December 2025 with the limits of each day. The limits move: they are narrow on busy days and wide on quiet ones.

In [ ]:
part = x.loc["2025-10-01":"2025-12-31"]
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(part.index, part.refund_rate * 100, lw=0.8, color="grey", marker=".", ms=3)
ax.step(part.index, ucl.loc[part.index] * 100, where="mid", color="black", lw=0.8, ls="--", label="limits of each day")
ax.step(part.index, lcl.loc[part.index] * 100, where="mid", color="black", lw=0.8, ls="--")
ax.axhline(pbar * 100, color="black", lw=0.8)
out = outside.loc[part.index]
ax.scatter(part.index[out], part.refund_rate[out] * 100, color="red", zorder=3, s=18, label="outside")
ax.set_ylabel("refund rate (%)")
ax.legend(loc="upper left")
plt.show()
print(x.loc["2025-11-28"])

Black Friday (2025-11-28) had 1,224 orders and a rate of 3.76%. Its upper limit that day was 3.38%, so the p-chart marks it. The I-MR chart has one limit for every day, and it does not.

## 3. p-chart or I-MR chart?

Run the next cell. It lists the days where the two charts disagree.

In [ ]:
center = p1.refund_rate.mean()
sigma_i = p1.refund_rate.diff().abs().mean() / 1.128
i_out = (x.refund_rate - center).abs() > 3 * sigma_i
print("only the p-chart:", [str(d.date()) for d in x.index[outside & ~i_out]])
print("only the I-MR chart:", [str(d.date()) for d in x.index[i_out & ~outside]])

The I-MR chart treats a day with 109 orders and a day with 1,372 orders the same. Use a p-chart when the volume of each day changes a lot. Use the I-MR chart when you have one number per day and no count behind it.

## 4. When the data vary more than the model says

The p-chart assumes that each order is refunded or not by chance, with the same chance on every day. Then z, the distance in sigmas, has a standard deviation of 1. Real processes have day effects too: a bad delivery day, a storm. Then z varies more than 1. This is **overdispersion**, and a plain p-chart gives too many alarms.

The **Laney p′-chart** (p-prime) measures the real spread of z from its moving range, as the I-MR chart does, and widens the limits by that factor, **sigma_z**.

> **Check.** You should see a standard deviation of z near 1.10 in phase I, and Laney sigma_z 1.121.

In [ ]:
z = (x.refund_rate - pbar) / sigma_day                    # z of the p-chart
print(f"phase I: standard deviation of z {z.loc[:'2024-12-31'].std():.3f} (1.000 if the simple model were right)")
laney = z.loc[:"2024-12-31"].diff().abs().mean() / 1.128   # sigma of z, from its moving range
print(f"Laney sigma_z {laney:.3f}")
laney_out = z.abs() > 3 * laney                           # the Laney p'-chart: limits 3 * sigma_z wider
print("Laney p', days outside:", [str(d.date()) for d in x.index[laney_out]])

With limits 1.121 times wider, the Laney chart keeps 4 of the 7 days of the p-chart. Here the overdispersion is small. The tickets below show a large one.

## 5. Counts: the c-chart

Sometimes we count events, and every day has the same chance to have them: complaints on a help desk with the same opening hours. For a count, the variance equals the mean, so one sigma is √c̄. This is the **c-chart**.

> **Check.** You should see c-bar 4.4 and day 8 outside.

In [ ]:
complaints = pd.Series([3, 5, 2, 4, 6, 3, 4, 11, 2, 4], index=pd.RangeIndex(1, 11, name="day"))
c_bar = complaints.mean()
c_ucl, c_lcl = c_bar + 3 * np.sqrt(c_bar), max(0, c_bar - 3 * np.sqrt(c_bar))   # a count: variance = mean
print(f"c-bar {c_bar}, limits {c_lcl:.2f} to {c_ucl:.2f}; outside: {list(complaints.index[complaints > c_ucl])}")

## 6. Counts per unit: the u-chart

When the "size" of each day changes, divide the count by it: tickets per order. One sigma is √(ū/n), with n the orders of the day. This is the **u-chart**. Run the next cell. It uses the daily orders and tickets of the shared series.

> **Check.** You should see u-bar 0.4785 and Laney sigma_z 3.57.

In [ ]:
orders = pd.read_csv("data/orders.csv").groupby("date").orders.sum()
tickets = pd.read_csv("data/tickets_daily.csv").groupby("date").tickets.sum()
u = pd.DataFrame({"orders": orders, "tickets": tickets}).dropna()
u = u[u.orders > 0]
u1 = u.loc["2024-04-08":"2024-12-31"]                     # phase I (the series's first week is left out)
u_bar = u1.tickets.sum() / u1.orders.sum()               # tickets per order
u["u"] = u.tickets / u.orders
uz = (u.u - u_bar) / np.sqrt(u_bar / u.orders)            # z of the u-chart
laney_u = uz.loc["2024-04-08":"2024-12-31"].diff().abs().mean() / 1.128
print(f"u-bar {u_bar:.4f}; {len(u)} days; u-chart alarm days {(uz.abs() > 3).sum()}; Laney sigma_z {laney_u:.2f}")
heat = uz.loc["2025-06-30":"2025-07-06"]
print(f"heat-wave week: u-chart alarms {(heat.abs() > 3).sum()} of 7 days, Laney u' {(heat.abs() > 3 * laney_u).sum()}")

The plain u-chart alarms on about half of the days. Tickets vary from day to day far more than a count model says, so its limits are too narrow. The Laney u′-chart widens them 3.57 times. In the heat-wave week, tickets AND orders rose, so the ratio stayed almost normal.

## 7. Check yourself

**Your turn 1.** What is the p-chart's upper limit on a day with 500 orders? Use `pbar`, round to 4 decimals, and write it as `ucl_500`.

In [ ]:
ucl_500 = round(pbar + 3 * np.sqrt(pbar * (1 - pbar) / 500), 4)

In [ ]:
expect('ucl_500', ucl_500, 0.0408, tolerance=0.0001)

**Your turn 2.** How many days of phase II (from 2025-01-01) are outside the p-chart's limits, on either side? Write the number as `phase2_outside`.

In [ ]:
phase2_outside = int(outside.loc["2025-01-01":].sum())

In [ ]:
expect('phase2_outside', phase2_outside, 6)

**Your turn 3.** How many days does the Laney u′-chart mark (|uz| above 3 × `laney_u`)? Write the number as `laney_u_days`.

In [ ]:
laney_u_days = int((uz.abs() > 3 * laney_u).sum())

In [ ]:
expect('laney_u_days', laney_u_days, 13)

## Recap

- A p-chart gives each day its own limits: narrow on a big day, wide on a small one. Black Friday's 3.76% was above its limit of 3.38%.
- A c-chart is for counts with the same chance every day; a u-chart is for counts per unit, such as tickets per order.
- When the data vary more than the model says (overdispersion), the Laney chart widens the limits by sigma_z: 1.121 for refunds, 3.57 for tickets.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Small, lasting shifts: CUSUM and EWMA](https://learning.nextia-ai.com/courses/anomaly-detection/m03/small-lasting-shifts-cusum-and-ewma/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m03/charts-for-rates-and-counts/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.